# Objetivo 1 (Operaciones CRUD).

Importamos las librerías necesarias.

In [11]:
!pip install -r ../requirements.txt --quiet

In [12]:
import redis
import pandas as pd
from tqdm import tqdm
from pprint import pprint

## Cargar CSV

Primero cargamos el CSV como un DataFrame de Pandas.

In [13]:
path = "../data/cards_final_with_xp.csv"
df = pd.read_csv(path)

### Comprobamos la estructura del Dataframe.

In [14]:
df.head() #Visualizamos las primera filas del DataFrame para comprobar que se ha cargado correctamente.

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic


In [15]:
print(list(df.columns)) # Estas son las columnas del DataFrame.

['code', 'name', 'text', 'type_code', 'traits', 'pack_code', 'illustrator', 'image_url', 'xp', 'faction_code']


In [16]:
len(df) # La longitud del Dataframe se corresponde con la cantidad de cartas que tenemos que gestionar.

5346

In [17]:
df.info(memory_usage="deep") #Comprobamos que lo que ocupa en memoria el df es razonable para trabajar con él en RAM.

<class 'pandas.DataFrame'>
RangeIndex: 5346 entries, 0 to 5345
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   code          5346 non-null   str  
 1   name          5346 non-null   str  
 2   text          5346 non-null   str  
 3   type_code     5346 non-null   str  
 4   traits        5346 non-null   str  
 5   pack_code     5346 non-null   str  
 6   illustrator   5346 non-null   str  
 7   image_url     5346 non-null   str  
 8   xp            5346 non-null   int64
 9   faction_code  5346 non-null   str  
dtypes: int64(1), str(9)
memory usage: 4.0 MB


<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 1</h2>

Observando la estructura de nuestro DataFrame concluimos que la mejor estructura de datos para almacenar esta información en Redis es un Hash. Con la siguiente estructura:


- *KEY*: "card:`code`"

- *FIELDS*: `code`, `name`, `text`, `type_code`, `traits`, `pack_code`,
       `illustrator`, `image_url`, `xp`, `faction_code`

- *VALUES*: Los valores correspondientes a cada campo en la fila del DataFrame

los motivos principales para elegir esta estructura son:

- Se define un namespace fijo que agrupa todas las cartas, y luego un identificador único para cada carta.
- Los podemos beneficiar de operaciones atómicas sobre los hashes como HGETALL, HSET, HGET, etc. que están optimizados para pares clave-valor como con los que estamos tra



### Creamos un CSV "de juguete".

Para ilustrar el proceso de cargar los datos vamos a definir un df "de juguete" con solo 10 filas.

In [18]:
df.sample(10).to_csv("sample_csv/s1.csv", index=False)

## Creamos la base de datos en REDIS.

Este comando lanza crea el contenedor de redis en el puerto 6379 y el puerto 8001 para acceder a la interfaz de redisinsight.

In [19]:
!docker run -d --name dread_db -p 6379:6379 -p 8001:8001 redis/redis-stack:latest

docker: Error response from daemon: Conflict. The container name "/dread_db" is already in use by container "19176615b8bd06a7b1062d89bde5360292ae405f0e00be15cb776b986aa69017". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


Nos conectamos a la base de datos con el cliente de Python redis-py.

In [20]:
r = redis.Redis(decode_responses=True)
r.flushall() # Limpiamos la base de datos para evitar conflictos con datos anteriores.

True

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 2</h2>

## CREATE

El cliente de python nos permite convertir un diccionario de Python en un hash mediante el parámetro mapping de la función hset. Por lo tanto, para cargar los datos del DataFrame en Redis, iteramos sobre cada fila del DataFrame, convertimos esa fila en un diccionario y luego usamos hset para almacenarla como un hash en Redis.

In [21]:
def create(file_path : str, redis_client : redis.Redis):

    try:
        redis_client.ping()
    except Exception as e:
        print(f"Error al conectar con Redis: {e}")
        return 0

    df = pd.read_csv(file_path)
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Cargando datos en Redis"):
        key = f"cards:{row['code']}"
        row_dict = row.to_dict()        
        redis_client.hset(key, mapping=row_dict)
    return 1

Test de la función de carga de datos.

In [22]:
path = "sample_csv/s1.csv"
ack = create(path, r)
if ack:
    print("Datos cargados correctamente en Redis.")
else:
    print("Error al cargar los datos en Redis.")

Cargando datos en Redis: 100%|██████████| 10/10 [00:00<00:00, 644.17it/s]

Datos cargados correctamente en Redis.


Comprobamos que los datos se cargan correctamente en Redis.

In [23]:
ex_keys = r.keys("cards:*")
print(f"Lista de claves en Redis: {ex_keys}")
print(f"hash de una de las cartas:")
pprint(r.hgetall(ex_keys[0]))

Lista de claves en Redis: ['cards:11525', 'cards:9085', 'cards:2019', 'cards:4069', 'cards:84035', 'cards:8717', 'cards:9554', 'cards:70034', 'cards:9542', 'cards:54004']
hash de una de las cartas:
{'code': '11525',
 'faction_code': 'mythos',
 'illustrator': 'Walter Brocca',
 'image_url': 'https://arkhamdb.com/bundles/cards/11525.jpg',
 'name': 'Treacherous Path',
 'pack_code': 'tdcc',
 'text': "X is this location's level. Forced - After Treacherous Path is "
         'revealed: Spawn the set-aside Coral Star Spawn enemy at this '
         'location, exhausted. It does not ready during the next upkeep phase.',
 'traits': "R'lyeh|Walkway",
 'type_code': 'location',
 'xp': '0'}


<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 3</h2>

Llegados a este punto, implementamos el resto de las operaciones CRUD que se nos han pedido (is_in, read, delete). Utilizando las funciones de redis-py para hashes. Añadiremos algo de lógica para el manejo de errores en caso de que la clave no exista.

In [24]:
def is_in(redis_client : redis.Redis, code : str) -> bool:
    key = f"cards:{code}"
    return redis_client.exists(key) == 1

def read(redis_client : redis.Redis, code : str) -> dict:
    key = f"cards:{code}"
    if redis_client.exists(key) == 0:
        print(f"La carta con código {code} no existe en Redis.")
        return {}
    return redis_client.hgetall(key)

def delete(redis_client : redis.Redis, code : str) -> bool:
    key = f"cards:{code}"
    if redis_client.exists(key) == 0:
        print(f"La carta con código {code} no existe en Redis.")
        return False
    redis_client.delete(key)
    return True

### Testeamos la funcionalidad de las operaciones CRUD implementadas.

In [25]:
ex_keys = r.keys("cards:*")
code_to_test_0 = ex_keys[0].split(":")[1]
code_to_test_1 = "123456" # Código que no existe en Redis.

print(f"¿La carta con código {code_to_test_0} existe en Redis? {is_in(r, code_to_test_0)}")
print(f"¿La carta con código {code_to_test_1} existe en Redis? {is_in(r, code_to_test_1)}")


print(f"\nDatos de la carta con código {code_to_test_0}:\n")
pprint(read(r, code_to_test_0))

print(f"\nEliminamos la carta con código {code_to_test_0}...")
if delete(r, code_to_test_0):
    print(f"Carta con código {code_to_test_0} eliminada correctamente.")
print(f"¿La carta con código {code_to_test_0} existe en Redis? {is_in(r, code_to_test_0)}")

¿La carta con código 2019 existe en Redis? True
¿La carta con código 123456 existe en Redis? False

Datos de la carta con código 2019:

{'code': '2019',
 'faction_code': 'guardian',
 'illustrator': 'John Pacer',
 'image_url': 'https://arkhamdb.com/bundles/cards/02019.jpg',
 'name': 'Taunt',
 'pack_code': 'dwl',
 'text': 'Fast. Play only during your turn. Engage any number of enemies at '
         'your location. Draw 1 card for each enemy engaged in this way.',
 'traits': 'Tactic',
 'type_code': 'event',
 'xp': '2'}

Eliminamos la carta con código 2019...
Carta con código 2019 eliminada correctamente.
¿La carta con código 2019 existe en Redis? False


Comprobamos que las funciones CRUD funcionan correctamente. Además en redis insights podemos ver todas las cartas que se han cargado y sus campos.

<img src="media/insights_0.png" width="600" style="display: block; margin: 0 auto; border: 2px solid white; border-radius: 10px;">

# Creamos una librería de las funciones CRUD.

para facilitar el uso de estas funciones las agrupamos en una librería. **Aprovechamos para hacer una demo de su funcionalidad y para cargar la base de datos completa**

In [26]:
from crud_handler import RedisCRUDHandler

r.flushall()

crud_handler = RedisCRUDHandler(r)
ack = crud_handler.create("../data/cards_final_with_xp.csv")

if ack:
    print("Datos cargados correctamente en Redis.")

Cargando datos en Redis:   0%|          | 0/5346 [00:00<?, ?it/s]

Cargando datos en Redis: 100%|██████████| 5346/5346 [00:04<00:00, 1107.20it/s]

Datos cargados correctamente en Redis.
